# 🎤 AICoverGen — seruvseruv-boop fork (Kaggle)

**Features:**
- ✅ RMVPE+ pitch algorithm (default)
- ✅ Toza minus (clean instrumental, backup_gain = -20)
- ✅ WAV output format (default)
- ✅ Index rate = 0.75 (default)
- ✅ Works on Kaggle (no HuggingFace Spaces required)

> ⚠️ **Muhim:** Notebook Settings → **Internet ON** va **GPU Accelerator (T4 GPU)** yoqing!

> 💡 Gradio public URL uchun `gradio_tunnel` yoki `ngrok` ishlatiladi.


In [ ]:
# ✅ Cell 1: GPU va muhit tekshirish
import subprocess, os

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print('✅ GPU topildi!')
    print(result.stdout[:300])
else:
    print('⚠️  GPU topilmadi! Notebook Settings → Accelerator → GPU (T4) tanlang!')

print('\n📂 Ish joyi:', os.getcwd())
print('💾 Disk:', subprocess.run(['df', '-h', '/'], capture_output=True, text=True).stdout.split('\n')[1])

In [ ]:
# ✅ Cell 2: Repo klonlash
import os

REPO_URL = 'https://github.com/seruvseruv-boop/AICoverGen.git'
REPO_DIR = '/kaggle/working/AICoverGen'

if os.path.exists(REPO_DIR):
    print('📁 Repo allaqachon bor, yangilanmoqda...')
    os.chdir(REPO_DIR)
    os.system('git pull origin main')
else:
    print('⬇️  Repo yuklab olinmoqda...')
    os.system(f'git clone {REPO_URL} {REPO_DIR}')
    os.chdir(REPO_DIR)

print('✅ Tayyor:', REPO_DIR)
os.listdir('.')

In [ ]:
# ✅ Cell 3: Sistema kutubxonalari
import os
print('📦 Tizim kutubxonalari o\'rnatilmoqda...')
os.system('apt-get update -qq')
os.system('apt-get install -y -qq aria2 ffmpeg sox')
print('✅ Tayyor!')

In [ ]:
# ✅ Cell 4: Python kutubxonalari o'rnatish
import os
print('🐍 Python kutubxonalari o\'rnatilmoqda (3-7 daqiqa ketishi mumkin)...')
ret = os.system('pip install -q -r requirements.txt')
os.system('python src/download_models.py')
if ret == 0:
    print('✅ Barcha kutubxonalar o\'rnatildi!')
else:
    print('⚠️  Ba\'zi xatolar bo\'lishi mumkin, lekin davom etishga harakat qilinadi...')

In [ ]:
# ✅ Cell 5: Cloudflare tunnel (public URL uchun)
import os, urllib.request, stat

CF_PATH = '/usr/local/bin/cloudflared'
if not os.path.exists(CF_PATH):
    print('⬇️  cloudflared yuklanmoqda...')
    urllib.request.urlretrieve(
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        CF_PATH
    )
    os.chmod(CF_PATH, stat.S_IRWXU | stat.S_IRGRP | stat.S_IXGRP | stat.S_IROTH | stat.S_IXOTH)
    print('✅ cloudflared o\'rnatildi!')
else:
    print('✅ cloudflared allaqachon bor!')

# gradio_tunnel_url ochish
try:
    import gradio as gr
    print('✅ Gradio versiyasi:', gr.__version__)
except:
    print('⚠️  Gradio yuklanmagan, Cell 4 ni qayta ishga tushiring')

In [ ]:
# ✅ Cell 6: AICoverGen ishga tushirish
# Gradio interfeysi — public URL chiqadi!
import os, sys, threading, subprocess, time

os.chdir('/kaggle/working/AICoverGen')
sys.path.insert(0, '/kaggle/working/AICoverGen/src')

print('🚀 AICoverGen ishga tushirilmoqda...')
print('📌 Bir necha soniyadan keyin public URL chiqadi')
print('='*60)

# Cloudflare tunnel bilan ishga tushirish
GRADIO_PORT = 7860

def run_tunnel():
    time.sleep(8)
    p = subprocess.Popen(
        ['/usr/local/bin/cloudflared', 'tunnel', '--url', f'http://localhost:{GRADIO_PORT}'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    for line in p.stdout:
        line = line.decode('utf-8', errors='ignore').strip()
        if 'trycloudflare.com' in line or '.trycloudflare.com' in line:
            # URL ni ajratib olish
            import re
            urls = re.findall(r'https://[\w-]+\.trycloudflare\.com', line)
            if urls:
                print('\n' + '='*60)
                print(f'🌐 PUBLIC URL: {urls[0]}')
                print('='*60 + '\n')

t = threading.Thread(target=run_tunnel, daemon=True)
t.start()

# webui ni ishga tushirish
os.system(f'python src/webui.py --listen --listen-port {GRADIO_PORT}')

In [ ]:
# ℹ️  Foydalanish yo'riqnomasi
print("""
=== AICoverGen Kaggle — Foydalanish ===

1. Yuqoridagi barcha celllarni tartib bilan ishga tushiring
2. Cell 6 dan PUBLIC URL ni oling
3. Brauzerda URL ni oching

=== Interfeys ichida ===
1. 'Generate' tabiga o'ting  
2. RVC model (.zip) URL ni kiriting
3. Qo'shiq URL (YouTube) ni kiriting
4. 'Generate' bosing — kutib turing!

=== Standart sozlamalar ===
- F0 Method    : rmvpe+  (eng aniq pitch)
- Index Rate   : 0.75
- Output Format: wav
- Backup Gain  : -20 dB  (toza minus)

=== Eslatma ===
Kaggle sessiyasi = 12 soat GPU, 30 soat/hafta
Internet = ON bo'lishi shart!
""")